# Diffusion Model Evaluation Pipeline

This notebook provides a standalone evaluation pipeline for class-conditional diffusion models. It is designed to work with any model checkpoint produced by the Lip-DP-Diffusion or DP-SGD training notebooks.

---

Given a saved model checkpoint, this notebook runs the full evaluation suite:

| Step | What happens |
|---|---|
| **1. Model Loading** | Instantiate `CondDiffUNet` and load weights from a `.pt` checkpoint. Supports both EMA and live weights. |
| **2. Image Generation** | Sample 20 000 images via DDPM reverse diffusion with classifier-free guidance (configurable scale $w$). |
| **3. FID / IS** | Compute Fréchet Inception Distance and Inception Score against 20 000 real MNIST training images using `torch-fidelity`. |
| **4. Downstream NN Accuracy** | Train a MLP and a CNN classifier on 60 000 synthetic images, evaluate on real MNIST test set. Measures whether generated images preserve class-discriminative structure. |
| **5. Downstream Classical Accuracy** | Train Logistic Regression, Decision Tree, and Random Forest on the same synthetic data.|
| **6. Results Summary** | All metrics collected into a single JSON and printed as a summary table. |

---

## Evaluation Metrics 

**Fréchet Inception Distance (FID):** Measures the distance between the distribution of generated images and real images in Inception feature space. Lower is better. A value of 0 would mean the two distributions are identical.

**Inception Score (IS):** Measures both the quality (low entropy per-image class predictions) and diversity (high entropy marginal class predictions) of generated images. Higher is better.

**Downstream Accuracy (Train-on-Synthetic, Test-on-Real):** Classifiers are trained exclusively on synthetic images and evaluated on real MNIST test images. This directly measures whether the synthetic data is useful as a privacy-preserving substitute for the real data. Higher accuracy means higher utility.

---

## How to Use

1. Set `CHECKPOINT_PATH` to your saved `.pt` file.
2. Set `HAS_EMA = True` if the checkpoint contains EMA weights (saved as `ema_shadow`), or `False` to use live model weights.
3. Set `GUIDANCE_SCALE` (0.0 = unconditional, 2.0 = strong class guidance).

### 1. Install & Imports

In [ ]:
import os, math, json, random
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.utils import save_image
from torch.utils.data import DataLoader

from torch_fidelity import calculate_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### 2. Configuration — Set These Before Running

In [ ]:
CHECKPOINT_PATH = "/path/to/your/checkpoint.pt"    # path to the model checkpoint .pt file

HAS_EMA = True    # True if checkpoint has EMA shadow weights
GUIDANCE_SCALE = 2.0     # classifier-free guidance scale (0.0 = unconditional)
BASE_CHANNELS = 32      # must match the training config
N_CLASSES = 11      # 10 digit classes + 1 null class

FID_IS_N = 20000   # images for FID/IS
SYNTH_N  = 60000   # images for downstream classifiers
BATCH_SIZE  = 256
DATASET = "mnist"   # "mnist" or "fashion_mnist"

OUT_DIR = "./eval_results"
os.makedirs(OUT_DIR, exist_ok=True)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

### 3. Mount Drive (Colab only)

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab, skipping drive mount.")

### 4. Load MNIST

In [ ]:
train_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),
])
test_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),
])

if DATASET == "fashion_mnist":
    mnist_train = FashionMNIST(root="./data", train=True,  transform=tfm, download=True)
    mnist_test  = FashionMNIST(root="./data", train=False, transform=tfm, download=True)
    print("Loaded Fashion-MNIST")
else:
    mnist_train = MNIST(root="./data", train=True,  transform=tfm, download=True)
    mnist_test  = MNIST(root="./data", train=False, transform=tfm, download=True)
    print("Loaded MNIST")

print(f"Train: {len(mnist_train)} | Test: {len(mnist_test)}")

### 5. Model Architecture

In [ ]:
class LipSiLU(nn.Module):
    def __init__(self, scale: float = 1/1.0998393):
        super().__init__()
        self.scale = float(scale)
    def forward(self, x):
        return self.scale * F.silu(x)


class LipschitzGroupNorm(nn.Module):
    def __init__(self, num_groups, num_channels):
        super().__init__()
        self.num_groups   = num_groups
        self.num_channels = num_channels

    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        x = x.view(B, G, C // G, H, W)
        mean  = x.mean(dim=[2, 3, 4], keepdim=True)
        var   = x.var(dim=[2, 3, 4],  keepdim=True, unbiased=False)
        denom = torch.clamp((var + 1e-5).sqrt(), min=1.0)
        return ((x - mean) / denom).view(B, C, H, W)

In [ ]:
def get_time_embedding(time_steps, t_emb_dim, device):
    half_dim = t_emb_dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
    emb = time_steps[:, None].float() * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if t_emb_dim % 2 == 1:
        emb = F.pad(emb, (0, 1, 0, 0))
    return emb

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
    def forward(self, x):
        return self.conv(self.up(x))

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, use_lip_norm=True):
        super().__init__()
        self.act    = LipSiLU()
        self.conv1  = nn.Conv2d(in_channels,  out_channels, 3, padding=1)
        self.conv2  = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.time_mlp = nn.Sequential(self.act, nn.Linear(t_emb_dim, out_channels * 2))
        self.skip_conv = (nn.Conv2d(in_channels, out_channels, 1)
                          if in_channels != out_channels else nn.Identity())
        if use_lip_norm:
            self.gn1 = LipschitzGroupNorm(8, out_channels)
            self.gn2 = LipschitzGroupNorm(8, out_channels)
        else:
            self.gn1 = nn.GroupNorm(8, out_channels, affine=False)
            self.gn2 = nn.GroupNorm(8, out_channels, affine=False)

    def forward(self, x, emb):
        h = self.gn1(self.conv1(x))
        time_emb    = self.time_mlp(emb).unsqueeze(-1).unsqueeze(-1)
        scale, shift = time_emb.chunk(2, dim=1)
        scale = (torch.tanh(scale) + 1) / 2
        h = self.act(h * scale + shift)
        h = self.act(self.gn2(self.conv2(h)))
        return (h + self.skip_conv(x)) / math.sqrt(2.0)

In [ ]:
class CondDiffUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, t_emb_dim=128,
                 n_classes=11, use_lip_norm=True):
        super().__init__()
        C = base_channels
        self.t_emb_dim = t_emb_dim
        self.n_classes = n_classes
        self.label_emb = nn.Embedding(n_classes, t_emb_dim)
        act = LipSiLU()
        self.time_proj = nn.Sequential(
            nn.Linear(t_emb_dim, t_emb_dim), act, nn.Linear(t_emb_dim, t_emb_dim))
        self.down1  = ResBlock(in_channels, C,    t_emb_dim, use_lip_norm)
        self.pool1  = nn.Conv2d(C,    C,    4, stride=2, padding=1)
        self.down2  = ResBlock(C,    C*2,  t_emb_dim, use_lip_norm)
        self.pool2  = nn.Conv2d(C*2, C*2, 4, stride=2, padding=1)
        self.down3  = ResBlock(C*2, C*4,  t_emb_dim, use_lip_norm)
        self.bot1   = ResBlock(C*4, C*4,  t_emb_dim, use_lip_norm)
        self.up1    = UpBlock(C*4, C*2)
        self.up_res1 = ResBlock(C*4, C*2, t_emb_dim, use_lip_norm)
        self.up2    = UpBlock(C*2, C)
        self.up_res2 = ResBlock(C*2, C,  t_emb_dim, use_lip_norm)
        self.out_conv = nn.Conv2d(C, in_channels, 3, padding=1)

    def forward(self, x, t, y):
        emb = self.time_proj(get_time_embedding(t, self.t_emb_dim, x.device)
                             + self.label_emb(y))
        x1 = self.down1(x, emb)
        x2 = self.down2(self.pool1(x1), emb)
        x3 = self.down3(self.pool2(x2), emb)
        bn = self.bot1(x3, emb)
        u1 = self.up_res1(torch.cat([self.up1(bn), x2], 1) / math.sqrt(2.0), emb)
        u2 = self.up_res2(torch.cat([self.up2(u1), x1], 1) / math.sqrt(2.0), emb)
        return self.out_conv(u2)

### 6. DDPM Schedule

In [ ]:
class DDPMConfig:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.T          = timesteps
        self.beta_start = beta_start
        self.beta_end   = beta_end

def make_ddpm_schedule(cfg, device):
    betas      = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.T, device=device)
    alphas     = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

@torch.no_grad()
def v_to_eps_and_x0(x_t, t, v, alpha_bars):
    ab         = alpha_bars[t].view(-1, 1, 1, 1)
    sqrt_ab    = torch.sqrt(ab)
    sqrt_1mab  = torch.sqrt(1.0 - ab)
    x0  = sqrt_ab * x_t - sqrt_1mab * v
    eps = sqrt_1mab * x_t + sqrt_ab  * v
    return eps, x0

ddpm_cfg = DDPMConfig(timesteps=1000, beta_start=1e-4, beta_end=0.02)
betas, alphas, alpha_bars = make_ddpm_schedule(ddpm_cfg, device)
print("DDPM schedule ready.")


### 7. EMA Helper

In [ ]:
class EMA:
    def __init__(self, model: nn.Module, decay=0.999):
        self.decay  = decay
        self.shadow = {n: p.detach().clone()
                       for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if not p.requires_grad: continue
            self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

    @torch.no_grad()
    def copy_to(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow:
                p.data.copy_(self.shadow[n].data)


### 8. Load Checkpoint

In [ ]:
model = CondDiffUNet(
    in_channels=1,
    base_channels=BASE_CHANNELS,
    n_classes=N_CLASSES,
    use_lip_norm=False,   # set to True if trained with LipGroupNorm
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

if HAS_EMA and "ema_shadow" in ckpt:
    # load EMA weights directly into model parameters
    state = model.state_dict()
    for name, param in ckpt["ema_shadow"].items():
        if name in state:
            state[name].copy_(param)
    model.load_state_dict(state)
    print("Loaded EMA weights from checkpoint.")
elif "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded live model weights from checkpoint.")
else:
    # assume the checkpoint is just the state_dict
    model.load_state_dict(ckpt)
    print("Loaded state_dict directly from checkpoint.")

# build a dummy EMA object wrapping the already-loaded weights
ema = EMA(model, decay=0.999)

model.eval()
print("Model ready.")


### 9. Generation Functions

In [ ]:
@torch.no_grad()
def generate_images(model, cfg, alphas, alpha_bars, betas,
                    n_samples, batch_size=256, guidance_scale=2.0):
    """
    Run full DDPM reverse diffusion and return images as uint8 tensors.
    Returns: images (N,1,28,28) uint8, labels (N,) int
    """
    model.eval()
    img_list, lab_list = [], []
    n_done = 0

    for _ in tqdm(range((n_samples + batch_size - 1) // batch_size),
                  desc=f"Generating (w={guidance_scale})"):
        cur = min(batch_size, n_samples - n_done)
        if cur <= 0: break

        x = torch.randn(cur, 1, 28, 28, device=device)
        y = torch.randint(0, 10, (cur,), device=device, dtype=torch.long)
        y_null = torch.full_like(y, 10)

        for i in reversed(range(cfg.T)):
            t = torch.full((cur,), i, device=device, dtype=torch.long)
            if guidance_scale > 0.0:
                v_cond   = model(x, t, y)
                v_uncond = model(x, t, y_null)
                v_pred   = v_uncond + guidance_scale * (v_cond - v_uncond)
            else:
                v_pred = model(x, t, y)

            eps_pred, _ = v_to_eps_and_x0(x, t, v_pred, alpha_bars)
            mean = (1.0 / torch.sqrt(alphas[i])) * (
                x - (betas[i] / torch.sqrt(1.0 - alpha_bars[i])) * eps_pred)
            x = mean + (torch.sqrt(betas[i]) * torch.randn_like(x) if i > 0 else 0.0)

        x01   = (x.clamp(-1, 1) + 1) / 2
        x_u8  = (x01 * 255.0).round().to(torch.uint8).cpu()
        img_list.append(x_u8)
        lab_list.append(y.cpu())
        n_done += cur

    return torch.cat(img_list)[:n_samples], torch.cat(lab_list)[:n_samples]

In [ ]:
def save_png_folder(images_u8, out_dir):
    """Save (N,1,28,28) uint8 tensor as individual PNGs for FID/IS."""
    os.makedirs(out_dir, exist_ok=True)
    for i, img in enumerate(tqdm(images_u8, desc="Saving PNGs")):
        save_image(img.float() / 255.0, os.path.join(out_dir, f"gen_{i:06d}.png"))


def prepare_real_images_folder(dataset, output_folder, num_images=20000):
    os.makedirs(output_folder, exist_ok=True)
    existing = len([f for f in os.listdir(output_folder) if f.endswith(".png")])
    if existing >= num_images:
        print(f"Real image folder already has {existing} images, skipping.")
        return output_folder
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    for i, (img, _) in enumerate(tqdm(loader, total=num_images, desc="Saving real images")):
        if i >= num_images: break
        save_image((img.squeeze(0) + 1) / 2,
                   os.path.join(output_folder, f"real_{i:06d}.png"))
    return output_folder


### 10. FID / IS

In [ ]:
def compute_fid_and_is(gen_dir, real_dir):
    metrics = calculate_metrics(
        input1=gen_dir, input2=real_dir,
        cuda=torch.cuda.is_available(),
        fid=True, isc=True, verbose=False,
    )
    return {
        "FID":     float(metrics["frechet_inception_distance"]),
        "IS_mean": float(metrics["inception_score_mean"]),
        "IS_std":  float(metrics["inception_score_std"]),
    }

### 11. Downstream Classifiers

In [ ]:
# Dataset wrapper for saved .pt synthetic data
class SynthMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, images_u8, labels):
        self.images_u8 = images_u8
        self.labels    = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x = self.images_u8[idx].float() / 255.0
        return (x * 2) - 1, self.labels[idx].long()

In [ ]:
# NN classifiers 
class MLPClassifier(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 10),
        )
    def forward(self, x): return self.net(x.view(x.size(0), -1))


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x):
        return self.fc(self.conv(x).view(x.size(0), -1))

In [ ]:
@torch.no_grad()
def eval_accuracy(clf, loader):
    clf.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (clf(x).argmax(1) == y).sum().item()
        total   += y.numel()
    return correct / total


def train_classifier(clf, train_loader, test_loader, epochs=10, lr=1e-3):
    clf = clf.to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=lr)
    ce  = nn.CrossEntropyLoss()
    for ep in range(1, epochs + 1):
        clf.train()
        for x, y in tqdm(train_loader, desc=f"[clf] ep {ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            loss = ce(clf(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return eval_accuracy(clf, test_loader)

In [ ]:
def downstream_eval_nn(synth_ds, mnist_test_ds, out_dir, batch_size=256):
    os.makedirs(out_dir, exist_ok=True)
    synth_loader = DataLoader(synth_ds,      batch_size=batch_size, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(mnist_test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    results = {
        "mlp": train_classifier(MLPClassifier(), synth_loader, test_loader),
        "cnn": train_classifier(SimpleCNN(),     synth_loader, test_loader),
    }
    save_json(results, os.path.join(out_dir, "downstream_nn.json"))
    return results


# Classical sklearn classifiers 
def downstream_eval_sklearn(synth_ds, mnist_test_ds, out_dir, random_state=42):
    os.makedirs(out_dir, exist_ok=True)

    # flatten to numpy
    X_tr = synth_ds.images_u8.numpy().astype(np.float32).reshape(len(synth_ds), -1) / 255.0
    y_tr = synth_ds.labels.numpy().astype(np.int64)

    loader = DataLoader(mnist_test_ds, batch_size=512, shuffle=False)
    X_te_list, y_te_list = [], []
    for x, y in loader:
        X_te_list.append(((x + 1) / 2).numpy().reshape(x.size(0), -1))
        y_te_list.append(y.numpy())
    X_te = np.concatenate(X_te_list); y_te = np.concatenate(y_te_list)

    results = {}
    print("[sklearn] LogisticRegression...")
    lr_clf = LogisticRegression(max_iter=1000, solver="lbfgs", n_jobs=-1)
    lr_clf.fit(X_tr, y_tr)
    results["logreg"] = float(accuracy_score(y_te, lr_clf.predict(X_te)))
    print("  acc:", results["logreg"])

    print("[sklearn] DecisionTree...")
    dt = DecisionTreeClassifier(random_state=random_state)
    dt.fit(X_tr, y_tr)
    results["decision_tree"] = float(accuracy_score(y_te, dt.predict(X_te)))
    print("  acc:", results["decision_tree"])

    print("[sklearn] RandomForest...")
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=random_state)
    rf.fit(X_tr, y_tr)
    results["random_forest"] = float(accuracy_score(y_te, rf.predict(X_te)))
    print("  acc:", results["random_forest"])

    save_json(results, os.path.join(out_dir, "downstream_sklearn.json"))
    return results

### 12. Run Full Evaluation

In [ ]:
#  1. Generate images 
print(" Step 1/4: Generating images")
images_u8, labels = generate_images(
    model, ddpm_cfg, alphas, alpha_bars, betas,
    n_samples=max(FID_IS_N, SYNTH_N),
    batch_size=BATCH_SIZE,
    guidance_scale=GUIDANCE_SCALE,
)
print(f" Generated {len(images_u8)} images.")

# split: FID_IS_N for FID/IS, SYNTH_N for downstream
fid_images = images_u8[:FID_IS_N]
synth_ds   = SynthMNISTDataset(images_u8[:SYNTH_N], labels[:SYNTH_N])

#  2. FID / IS 
print("\n Step 2/4: FID / IS")
gen_dir  = os.path.join(OUT_DIR, f"gen_pngs_w{GUIDANCE_SCALE}")
real_dir = os.path.join(OUT_DIR, "real_pngs")

save_png_folder(fid_images, gen_dir)
prepare_real_images_folder(mnist_train, real_dir, num_images=FID_IS_N)
fid_is = compute_fid_and_is(gen_dir, real_dir)
print(f" FID: {fid_is['FID']:.2f}")
print(f" IS mean: {fid_is['IS_mean']:.3f} ± {fid_is['IS_std']:.3f}")

#  3. Downstream NN 
print("\n Step 3/4: Downstream NN classifiers")
nn_res = downstream_eval_nn(synth_ds, mnist_test, OUT_DIR)
print(f"MLP acc: {nn_res['mlp']:.4f}")
print(f"CNN acc: {nn_res['cnn']:.4f}")

#  4. Downstream sklearn 
print("\n Step 4/4: Downstream classical classifiers")
sk_res = downstream_eval_sklearn(synth_ds, mnist_test, OUT_DIR)
print(f" LogReg acc: {sk_res['logreg']:.4f}")
print(f" DecisionTree acc:{sk_res['decision_tree']:.4f}")
print(f" RandomForest acc:{sk_res['random_forest']:.4f}")

### 13. Results Summary

In [ ]:
combined = {
    "checkpoint": CHECKPOINT_PATH,
    "guidance_scale":  GUIDANCE_SCALE,
    "has_ema": HAS_EMA,
    "FID":  fid_is["FID"],
    "IS_mean": fid_is["IS_mean"],
    "IS_std": fid_is["IS_std"],
    "downstream_nn": nn_res,
    "downstream_sklearn": sk_res,
}
save_json(combined, os.path.join(OUT_DIR, "all_metrics.json"))

print(" Evaluation summary:")
print(f" Checkpoint: {os.path.basename(CHECKPOINT_PATH)}")
print(f" Guidance scale: w = {GUIDANCE_SCALE}")
print(f" FID: {fid_is['FID']:.2f}  (lower is better)")
print(f" IS:  {fid_is['IS_mean']:.3f} ± {fid_is['IS_std']:.3f}  (higher is better)")
print(f" Downstream MLP: {nn_res['mlp']*100:.1f}%")
print(f" Downstream CNN: {nn_res['cnn']*100:.1f}%")
print(f" Downstream LogReg: {sk_res['logreg']*100:.1f}%")
print(f" Downstream RF: {sk_res['random_forest']*100:.1f}%")
print("="*50)
print(f"\nAll results saved to: {OUT_DIR}/all_metrics.json")